<a href="https://colab.research.google.com/github/hoanbklucky/duckietown-lx/blob/mooc2022/working_dt_object_detection_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# First, let us set up a few dependencies

Don't forget to switch to a GPU-enabled colab runtime!

```
Runtime -> Change Runtime Type -> GPU
```

In [17]:
import os
import contextlib
@contextlib.contextmanager
def directory(name):
  ret = os.getcwd()
  os.chdir(name)
  yield None
  os.chdir(ret)

import subprocess
def run(input, exception_on_failure=False):
  try:
    program_output = subprocess.check_output(f"{input}", shell=True, universal_newlines=True, stderr=subprocess.STDOUT)
  except Exception as e:
    if exception_on_failure:
      raise e
    program_output = e.output

    return program_output
def prun(input, exception_on_failure=False):
  x = run(input, exception_on_failure)
  print(x)
  return x

# This mounts your google drive to this notebook. You might have to change the path to fit with your dataset folder inside your drive.

Read the instruction output by the cell bellow carefully!

In [18]:
# Create a temporary workspace
import tempfile


SESSION_WORKSPACE = tempfile.mkdtemp()
print(f"Session workspace created at: {SESSION_WORKSPACE}")

Session workspace created at: /tmp/tmpahn4bh07


In [19]:
# Mount the drive
from google.colab import drive
drive.mount('/content/drive')
DRIVE_PATH = "/content/drive/My Drive"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [20]:
# Unzip the dataset
import shutil
import os


DATASET_DIR_NAME = "duckietown_object_detection_dataset"
DATASET_ZIP_NAME = f"{DATASET_DIR_NAME}.zip"
DATASET_DIR_PATH = os.path.join(SESSION_WORKSPACE, DATASET_DIR_NAME)
TRAIN_DIR = "train"
VALIDATION_DIR = "val"
IMAGES_DIR = "images"
LABELS_DIR = "labels"


def show_info(base_path: str):
  for l1 in [TRAIN_DIR, VALIDATION_DIR]:
    for l2 in [IMAGES_DIR, LABELS_DIR]:
      p = os.path.join(base_path, l1, l2)
      print(f"#Files in {l1}/{l2}: {len(os.listdir(p))}")


def unzip_dataset():
  # check zipped file
  zip_path = os.path.join(DRIVE_PATH, DATASET_ZIP_NAME)
  assert os.path.exists(zip_path), f"No zipped dataset found at {zip_path}! Abort!"

  # unzip the data
  print("Unpacking zipped data...")
  shutil.unpack_archive(zip_path, DATASET_DIR_PATH)
  print(f"Zipped dataset unpacked to {DATASET_DIR_PATH}")

  # show some info
  show_info(DATASET_DIR_PATH)


unzip_dataset()

Unpacking zipped data...
Zipped dataset unpacked to /tmp/tmpahn4bh07/duckietown_object_detection_dataset
#Files in train/images: 964
#Files in train/labels: 964
#Files in val/images: 242
#Files in val/labels: 242


In [21]:
# change  working directory to the session workspace
os.chdir(SESSION_WORKSPACE)
print(f"PWD: {os.getcwd()}")

# install pytorch and torchvision
#!pip3 install torch==1.13.0 torchvision==0.14.0
!pip3 install torch torchvision

PWD: /tmp/tmpahn4bh07


# Next, we will clone Yolov5

In [22]:
!rm -rf ./yolov5
!git clone https://github.com/ultralytics/yolov5.git -b v7.0
!cd yolov5 && pip3 install -r requirements.txt

Cloning into 'yolov5'...
remote: Enumerating objects: 17726, done.
remote: Counting objects: 100% (98/98), done.
remote: Compressing objects: 100% (73/73), done.
remote: Total 17726 (delta 37), reused 27 (delta 25), pack-reused 17628 (from 3)
Receiving objects: 100% (17726/17726), 17.20 MiB | 19.17 MiB/s, done.
Resolving deltas: 100% (12016/12016), done.
Note: switching to '915bbf294bb74c859f0b41f1c23bc395014ea679'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false



In [23]:
!rm -rf yolov5
!git clone https://github.com/ultralytics/yolov5
%cd yolov5
!pip install -r requirements.txt

Cloning into 'yolov5'...
remote: Enumerating objects: 17726, done.
remote: Counting objects: 100% (98/98), done.
remote: Compressing objects: 100% (73/73), done.
remote: Total 17726 (delta 37), reused 27 (delta 25), pack-reused 17628 (from 3)
Receiving objects: 100% (17726/17726), 17.22 MiB | 9.15 MiB/s, done.
Resolving deltas: 100% (12024/12024), done.
/tmp/tmpahn4bh07/yolov5


In [24]:
%%writefile data/duckietown.yaml
# train and val data paths
train: ../duckietown_object_detection_dataset/train
val: ../duckietown_object_detection_dataset/val

# number of classes
nc: 4

# class names
names: [ 'duckie', 'cone', 'truck', 'bus' ]

Writing data/duckietown.yaml


# We now inform the training process of the format and location of our dataset

# And we're ready to train! This step will take about 5 minutes.

Notice that we're only training for 10 epochs. That's probably not enough!

In [25]:
!sed -i "s/ckpt = torch.load(weights, map_location='cpu')/import torch.serialization; import models.yolo; torch.serialization.add_safe_globals([models.yolo.Model]); ckpt = torch.load(weights, map_location='cpu', weights_only=False)/" train.py

!python train.py --cfg ./models/yolov5n.yaml --img 416 --batch 32 --epochs 100 --data data/duckietown.yaml --weights yolov5n.pt

Streaming output truncated to the last 5000 lines.
      26/99      2.04G     0.0368    0.01142   0.001879        143        416:  97% 30/31 [00:09<00:00,  2.43it/s]/tmp/tmpahn4bh07/yolov5/train.py:414: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(amp):
      26/99      2.04G    0.03662    0.01172   0.001956         38        416: 100% 31/31 [00:09<00:00,  3.12it/s]
                 Class     Images  Instances          P          R      mAP50   mAP50-95: 100% 4/4 [00:02<00:00,  1.51it/s]
                   all        242        662      0.937      0.897      0.941      0.522

      Epoch    GPU_mem   box_loss   obj_loss   cls_loss  Instances       Size
  0% 0/31 [00:00<?, ?it/s]/tmp/tmpahn4bh07/yolov5/train.py:414: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(amp):
      

In [36]:
import numpy as np
import os

# Change to the yolov5 directory
#os.chdir("yolov5")

all_exps = os.listdir("runs/train")
all_exps_filtered = map(lambda x: int(x.replace("exp", "1")), filter(lambda x: x.startswith("exp"), all_exps))
all_exps_filtered = np.array(list(all_exps))
latest_exp_index = np.argmax(all_exps)
latest_exp = all_exps[latest_exp_index]
print(f"Latest exp is {latest_exp}")

prun(f"cp runs/train/{latest_exp}/weights/best.pt best.pt")
print(f"Marked the model from the latest run ({latest_exp}) as yolov5/best.pt.")

# Change back to the session workspace directory
os.chdir("..")

Latest exp is exp
None
Marked the model from the latest run (exp) as yolov5/best.pt.


In [40]:
from google.colab import files

# Pfad zu deinem Modell
model_path = "yolov5/best.pt"

# Datei herunterladen
files.download(model_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Next, we can upload your model to Duckietown's cloud!

We will need our token to access our personal cloud space.

In [41]:
# TODO: Fill in the duckietown token here
YOUR_DT_TOKEN = "dt1-DHcEZkb4F7AbsdjUR4eQYFUjxQMos1NBd4s9EBNdc6HvXN4U-43dzqWFnWd8KBa1yev1g3UKnzVxZkkTbfhqFnRP3DrZPxAhFRNTht8wX69RJRekMAN"

Then, we chose the location of the trained model on disk and its name once uploaded to our cloud space. You should not change these values, or the robots will not be able to find the model to download.

In [42]:
import sys
sys.path.insert(0, './yolov5')

# DO NOT CHANGE THESE
model_name = "yolov5n"
model_local_path = "./yolov5/best.pt"
model_remote_path = f"courses/mooc/objdet/data/nn_models/{model_name}.pt"

# install DCSS client
!pip3 install dt-data-api

We now open a pointer to our cloud space and upload the model.

In [43]:
import torch
from dt_data_api import DataClient, Storage

# open a pointer to our personal duckietown cloud space
client = DataClient(YOUR_DT_TOKEN)
storage = client.storage("user")

# upload model
upload = storage.upload(model_local_path, model_remote_path)
upload.join()

# Done!

We're done training! You can now close this tab and go back to the `Training` notebook